# Simple real NISAR GUNW workflow

This notebook is intentionally small. Set `SNOWIN_GUNW` to a local NISAR GUNW product and optionally set `SNOWIN_NISAR_DEM` to a cached NISAR-modified Copernicus DEM. The product is opened first for inspection; the slower incidence calculation is an explicit second step.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from snowin import compute_dswe
from snowin.io import add_gunw_incidence, open_gunw
from snowin.io.gunw import read_attrs_hdf5, read_scalar_hdf5

gunw_text = os.environ.get("SNOWIN_GUNW")
if not gunw_text:
    gunw_text = input("Enter the path to a local NISAR GUNW product: ").strip()
if not gunw_text:
    raise RuntimeError("A local NISAR GUNW product path is required.")
GUNW_PATH = Path(gunw_text).expanduser().resolve()
DEM_PATH = os.environ.get("SNOWIN_NISAR_DEM")
DEM_CACHE = Path(os.environ.get("SNOWIN_DEM_CACHE", "~/.cache/snowin/dem")).expanduser()
print("GUNW:", GUNW_PATH)
print("Local NISAR DEM:", DEM_PATH or "automatic download/cache")

## Inspect product metadata

In [ ]:
phase_path = "science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH/unwrappedPhase"
height_path = "science/LSAR/GUNW/metadata/radarGrid/heightAboveEllipsoid"
reference_path = "science/LSAR/identification/referenceZeroDopplerStartTime"
secondary_path = "science/LSAR/identification/secondaryZeroDopplerStartTime"

print("phase attributes:", read_attrs_hdf5(GUNW_PATH, phase_path))
print("height attributes:", read_attrs_hdf5(GUNW_PATH, height_path))
print("reference time:", read_scalar_hdf5(GUNW_PATH, reference_path))
print("secondary time:", read_scalar_hdf5(GUNW_PATH, secondary_path))

## Open and inspect the GUNW with SnowIn

This step does not download a DEM or compute incidence. Phase/product arrays remain lazy when Dask is installed, so you can inspect metadata before starting the slower geometry step.

In [ ]:
pair = open_gunw(GUNW_PATH, chunks="auto")
print(pair)
print("phase transform:", pair.attrs.get("phase_transform"))
print("incidence status:", pair.attrs.get("incidence_angle_status"))
print("wavelength [m]:", pair.attrs.get("wavelength_m"))

## Add local incidence when ready

This explicit call requires the GUNW path and appends `incidence_angle` to the already-open dataset. It is the step that may download/reproject the NISAR DEM and read the GUNW LOS cube.

In [ ]:
incidence_kwargs = {"dem_cache_dir": DEM_CACHE, "progress": True}
if DEM_PATH:
    incidence_kwargs["nisar_cop30_dem"] = Path(DEM_PATH).expanduser()

add_gunw_incidence(pair, GUNW_PATH, **incidence_kwargs)
print("incidence status:", pair.attrs.get("incidence_angle_status"))
print("DEM source:", pair.attrs.get("dem_source"))

## Preview phase, local incidence, and pairwise dSWE

In [ ]:
step = max(1, min(pair.sizes.values()) // 600)
phase_preview = pair["phase"].isel(y=slice(None, None, step), x=slice(None, None, step))
incidence_preview = np.rad2deg(pair["incidence_angle"]).isel(y=slice(None, None, step), x=slice(None, None, step))
valid_geometry = (pair["incidence_angle"] >= 0) & (pair["incidence_angle"] < np.pi / 2)
safe_incidence = pair["incidence_angle"].where(valid_geometry)
dswe = compute_dswe(
    pair["phase"],
    safe_incidence,
    wavelength_m=float(pair.attrs["wavelength_m"]),
)
dswe_preview = dswe.isel(y=slice(None, None, step), x=slice(None, None, step))

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
phase_preview.plot(ax=axes[0], cmap="viridis")
axes[0].set_title("Canonical phase [rad]")
incidence_preview.plot(ax=axes[1], cmap="magma")
axes[1].set_title("Local incidence [deg]")
dswe_preview.plot(ax=axes[2], cmap="RdBu_r")
axes[2].set_title("Pairwise dSWE [m]")
plt.show()

In [ ]:
print("phase definition:", pair.attrs.get("phase_difference_definition"))
print("incidence reference:", pair["incidence_angle"].attrs.get("incidence_angle_reference"))
print("finite phase fraction:", float(np.isfinite(phase_preview).mean()))
print("finite dSWE fraction:", float(np.isfinite(dswe_preview).mean()))
pair.close()